# Product Type 1 - 불량여부 및 불량유형 예측 모델

## 환경설정

In [ ]:
import pandas as pd
import numpy as np
import platform
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from sklearn.model_selection import train_test_split

if platform.system() == 'Windows':
    plt.rcParams['font.family'] = 'Malgun Gothic'
elif platform.system() == 'Darwin':
    plt.rcParams['font.family'] = 'AppleGothic'
else:
    plt.rcParams['font.family'] = 'NanumGothic'
plt.rcParams['axes.unicode_minus'] = False

plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.figsize'] = (12, 6)

print("="*60)
print("라이브러리 로드 완료!")
print("한글 폰트 설정 완료!")
print("="*60)

## 데이터 로드

In [ ]:
product_type_1 = pd.read_csv('../data_processed/product_type_1.csv', header=[0, 1])

print("="*60)
print("product_type_2 데이터 로드 완료!")
print("="*60)

## 데이터 전처리

In [ ]:
# 결측치 확인
print("\n" + "="*60)
print("결측치 확인")
print("="*60)

missing_df = pd.DataFrame({
    '결측수': product_type_1.isnull().sum(),
    '결측비율(%)': (product_type_1.isnull().sum() / len(product_type_1) * 100).round(2)
})
missing_df = missing_df[missing_df['결측수'] > 0].sort_values('결측수', ascending=False)

if len(missing_df) > 0:
    print("\n[결측치 현황]")
    display(missing_df)
else:
    print("\n결측치 없음")

In [ ]:
# 결측치 중앙값 대체
na_cols = product_type_1.columns[product_type_1.isnull().any()]
product_type_1[na_cols] = product_type_1[na_cols].fillna(product_type_1[na_cols].median())

print("="*60)
print("중앙값 대체 완료!!")
print("="*60)
print(f"결측치 데이터 수: {product_type_1.isnull().any().sum()}개")
print("="*60)

## 불량유형 컬럼 범주화

In [ ]:
process_data = product_type_1['process']
sensor_data = product_type_1['sensor']

# X값 준비 - process, sensor 컬럼명 평탄화
X = pd.concat([process_data, sensor_data], axis=1)
X.columns = [f"process_{col}" for col in process_data.columns] + \
            [f"sensor_{col}" for col in sensor_data.columns]

# 누수 방지 컬럼 제거
leak_cols = ["process_shot", "defect_flag_is_defect"]
X = X.drop(columns=leak_cols, errors='ignore')

# y 값 준비
defects_data = product_type_1['defects']

# 컬럼 범주화
defect_groups = {
    "표면": [
        "stain_1", "stain_2",
        "dent_1", "dent_2",
        "scratch_1", "scratch_2",
        "buring_mark_1", "buring_mark_2",
        "contamination_1", "contamination_2",
        "exfoliation_1", "exfoliation_2",
    ],
    "구조": [
        "short_shot_1", "short_shot_2",
        "bubble_1", "bubble_2",
        "blow_hole_1", "blow_hole_2",
        "deformation_1", "deformation_2",
        "crack_1", "crack_2",
        "impurity_1", "impurity_2",
        "inclusions_1", "inclusions_2",
    ],
}

# 불량유형 범주화 (그룹 내 하나라도 불량이면 1)
y = pd.DataFrame(index=defects_data.index)
for group, cols in defect_groups.items():
    usable = [c for c in cols if c in defects_data.columns]
    y[group] = defects_data[usable].max(axis=1) if usable else 0

print("="*60)
print(y.value_counts().sort_index())
print("="*60)

# 'stratify' 분할 기준용 생성
strata = y.astype(str).agg("".join, axis=1)  # 분할 기준용
print("stratify 분할 기준 데이터:")
print("="*60)
print(strata.value_counts().sort_index())

## Train/Test 분리

In [ ]:
# train, test 데이터 분리
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=strata
)

print("="*60)
print(f"Train set: {X_train.shape}")
print(f"Test set: {X_test.shape}")

print("="*60)
print("\n[Train set 타겟 분포]")
print("="*60)
print(y_train.value_counts())

print("="*60)
print("\n[Test set 타겟 분포]")
print("="*60)
print(y_test.value_counts())

In [ ]:
# train.csv 파일 저장
X_train.to_csv('../data_processed/X_train_1.csv', index=False)
y_train.to_csv('../data_processed/y_train_1.csv', index=False)